# K-means Clustering: From First Principles

**Learning goal:** understand the K-means objective, implement it with NumPy, compare with scikit-learn, test edge cases, and reason about large data and GPU trade-offs.

This notebook focuses on K-means and K-means++ initialization. DBSCAN, hierarchical clustering, spectral clustering, BIRCH, and Gaussian mixture models make different assumptions and should be explored in separate notebooks.

## 1. Definition and intuition

K-means divides $n$ observations into $K$ clusters. Every observation is assigned to its nearest centroid, and each centroid is recomputed as the mean of the observations assigned to it. The objective is to minimize within-cluster squared Euclidean distance.

## 2. Objective and why the mean is the update

Let $x_i\in\mathbb{R}^d$, assignment $z_i\in\{1,\ldots,K\}$, and centroid $\mu_k$. K-means minimizes

$$J=\sum_{i=1}^{n}\|x_i-\mu_{z_i}\|_2^2.$$

**Assignment step:** if centroids are fixed, assigning each point to its nearest centroid minimizes that point's contribution to $J$.

**Update step:** with assignments fixed, cluster $k$'s contribution is $\sum_{i:z_i=k}\|x_i-\mu_k\|^2$. Differentiating with respect to $\mu_k$ and setting the derivative to zero yields

$$\mu_k=\frac{1}{n_k}\sum_{i:z_i=k}x_i,$$

which is the arithmetic mean. Therefore each exact alternating step cannot increase the objective. The objective is bounded below by zero, and there are finitely many possible assignments; under ordinary exact Lloyd iterations, the algorithm reaches a stable assignment, although ties and implementation details can affect the path. This is **not** a guarantee of a global optimum: initialization matters.

## 3. Assumptions and failure modes

K-means is most suitable when clusters are reasonably compact and separable under Euclidean distance. It is sensitive to feature scale and outliers, requires choosing $K$, and often performs poorly on non-convex shapes or clusters with very different densities. Standardize features when scale differences do not carry meaningful importance. Do not standardize blindly: feature units and domain meaning should guide the distance metric.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score

SEED = 42
rng = np.random.default_rng(SEED)

## 4. Toy data

We first use well-separated blobs, which approximately satisfy K-means' geometric preference. Later, we use crescent-shaped `moons` to show a failure mode.

In [ ]:
X, y_reference = make_blobs(
    n_samples=500, centers=4, n_features=2, cluster_std=0.75, random_state=SEED
)
plt.scatter(X[:, 0], X[:, 1], c=y_reference, s=18)
plt.title("Synthetic blobs (reference labels shown only for illustration)")
plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()

## 5. NumPy implementation

The squared-distance identity

$$\|x-\mu\|^2=\|x\|^2+\|\mu\|^2-2x^T\mu$$

allows matrix-based distance calculation without constructing a huge $n\times K\times d$ tensor. We chunk the rows so temporary distance storage is limited. Empty clusters are reinitialized to a high-error point; this is a practical policy, not a unique mathematical requirement.

In [ ]:
def kmeans_numpy(X, n_clusters, max_iter=100, tol=1e-4, random_state=42, batch_size=8192):
    """Educational Lloyd K-means with chunked assignments and empty-cluster recovery."""
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError("X must be a non-empty 2D array")
    if not np.isfinite(X).all():
        raise ValueError("X must contain only finite values")
    n, d = X.shape
    if not isinstance(n_clusters, (int, np.integer)) or not 1 <= n_clusters <= n:
        raise ValueError("n_clusters must be an integer in [1, n_samples]")
    if max_iter < 1 or tol < 0 or batch_size < 1:
        raise ValueError("max_iter and batch_size must be positive; tol must be nonnegative")

    rng = np.random.default_rng(random_state)
    centroids = X[rng.choice(n, size=n_clusters, replace=False)].copy()
    x_sq = np.einsum("ij,ij->i", X, X)

    def assign_all(C):
        labels = np.empty(n, dtype=np.int64)
        min_dist = np.empty(n, dtype=np.float64)
        c_sq = np.einsum("ij,ij->i", C, C)
        for start in range(0, n, batch_size):
            stop = min(start + batch_size, n)
            dist = x_sq[start:stop, None] + c_sq[None, :] - 2 * X[start:stop] @ C.T
            np.maximum(dist, 0, out=dist)  # suppress tiny negative round-off
            local_labels = dist.argmin(axis=1)
            labels[start:stop] = local_labels
            min_dist[start:stop] = dist[np.arange(stop - start), local_labels]
        return labels, min_dist

    for iteration in range(max_iter):
        labels, errors = assign_all(centroids)
        new_centroids = centroids.copy()
        for j in range(n_clusters):
            members = X[labels == j]
            if len(members):
                new_centroids[j] = members.mean(axis=0)
            else:
                new_centroids[j] = X[np.argmax(errors)]
        shift = np.linalg.norm(new_centroids - centroids)
        centroids = new_centroids
        if shift <= tol:
            break

    labels, errors = assign_all(centroids)
    info = {"inertia": float(errors.sum()), "n_iter": iteration + 1}
    return centroids, labels, info

In [ ]:
centroids, labels, info = kmeans_numpy(X, n_clusters=4, random_state=SEED)
print(info)
plt.scatter(X[:, 0], X[:, 1], c=labels, s=18)
plt.scatter(centroids[:, 0], centroids[:, 1], marker="X", s=180, edgecolor="black")
plt.title("K-means implemented with NumPy")
plt.show()

## 6. Compare with scikit-learn

Cluster labels are arbitrary: cluster `0` from one model can correspond to cluster `2` from another. Compare objective values and label-invariant metrics rather than requiring integer labels to be identical.

In [ ]:
sk_model = KMeans(n_clusters=4, init="k-means++", n_init=10, random_state=SEED).fit(X)
print("NumPy inertia:", info["inertia"])
print("scikit-learn inertia:", float(sk_model.inertia_))
print("ARI vs known synthetic labels (NumPy):", adjusted_rand_score(y_reference, labels))
print("ARI vs known synthetic labels (sklearn):", adjusted_rand_score(y_reference, sk_model.labels_))
assert np.isfinite(centroids).all() and np.isfinite(info["inertia"])
assert len(labels) == len(X)

## 7. K-means++ initialization

Random initialization can place several centers near each other and leave another region without an initial center. K-means++ chooses the first center at random, then chooses each next center with probability proportional to squared distance from the nearest already chosen center. The expected initial potential has a logarithmic-factor approximation guarantee relative to the optimum; this improves initialization but does not make the subsequent Lloyd iterations globally optimal.

In [ ]:
def kmeans_plus_plus_init(X, k, random_state=42):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2 or len(X) == 0 or not 1 <= k <= len(X):
        raise ValueError("X must be 2D and k must be in [1, n_samples]")
    if not np.isfinite(X).all():
        raise ValueError("X must contain only finite values")
    rng = np.random.default_rng(random_state)
    centers = [X[rng.integers(len(X))].copy()]
    closest_sq = np.sum((X - centers[0]) ** 2, axis=1)
    for _ in range(1, k):
        total = closest_sq.sum()
        if total <= 0:
            # All points may be identical; any sample is equivalent geometrically.
            idx = rng.integers(len(X))
        else:
            idx = rng.choice(len(X), p=closest_sq / total)
        centers.append(X[idx].copy())
        new_dist = np.sum((X - centers[-1]) ** 2, axis=1)
        closest_sq = np.minimum(closest_sq, new_dist)
    return np.asarray(centers)

initial_centers = kmeans_plus_plus_init(X, 4, random_state=SEED)
print("initial center array shape:", initial_centers.shape)

## 8. Hyperparameters and best practices

- `n_clusters`: the number of clusters, chosen by the user. Use domain knowledge, an inertia curve, silhouette score on a manageable sample, and stability checks together rather than trusting one metric.
- `init`: `k-means++` is generally a sensible default in mature implementations.
- `n_init`: the number of independent initializations; more restarts cost time but can reduce poor-local-minimum risk.
- `max_iter`: maximum Lloyd iterations.
- `tol`: stopping threshold; details vary by implementation.
- `batch_size`: in our implementation, controls assignment-distance temporary memory.
- Inspect feature scales and outliers. Use multiple restarts for important results.
- MiniBatch K-means trades exact Lloyd updates for approximate updates and may be more practical on very large datasets.

## 9. Time and space complexity

Let $n$=samples, $d$=features, $K$=clusters, $T$=iterations, and $B$=chunk size.

| Operation | Time | Space notes |
|---|---:|---|
| One assignment pass | $O(nKd)$ | Model $O(Kd)$, plus $O(BK)$ temporary distance matrix with chunking |
| Centroid update | $O(nd)$ | Labels/errors are $O(n)$; summation can be optimized with grouped reductions |
| Lloyd training | approximately $O(TnKd)$ | Input itself is $O(nd)$ when dense |
| Predict $m$ new rows | $O(mKd)$ | Chunking reduces temporary space |

A naive broadcasted tensor of shape $(n,K,d)$ would require $O(nKd)$ intermediate storage. This notebook avoids that tensor. The exact runtime depends on initialization, dimensions, iteration count, memory bandwidth, and BLAS implementation.

## 10. Scaling to millions of samples

- Keep assignment distances chunked; do not materialize an $n\times K\times d$ tensor.
- Consider float32 when its numerical accuracy is adequate. A dense float64 array of one million rows and 100 features occupies about 800 MB before intermediate arrays.
- Use `MiniBatchKMeans` when approximate updates are acceptable.
- Benchmark wall-clock time, peak RAM, inertia, and external/validation quality. Do not compare implementations using different initializations without acknowledging it.
- Silhouette score can itself be expensive on large datasets; compute it on a representative sample when appropriate.

In [ ]:
def benchmark_kmeans(n_samples, n_features=8, n_clusters=8, seed=0):
    Xb, _ = make_blobs(n_samples=n_samples, centers=n_clusters,
                       n_features=n_features, random_state=seed)
    model = KMeans(n_clusters=n_clusters, n_init=3, random_state=seed)
    start = time.perf_counter()
    model.fit(Xb)
    elapsed = time.perf_counter() - start
    return {"n_samples": n_samples, "n_features": n_features,
            "fit_seconds": round(elapsed, 3),
            "inertia": round(float(model.inertia_), 2),
            "iterations": int(model.n_iter_)}

# A modest benchmark first. Increase sizes only if RAM is sufficient.
[benchmark_kmeans(n) for n in (1_000, 10_000, 100_000)]

In [ ]:
# Compare conventional Lloyd K-means and MiniBatch K-means on a larger set.
X_large, _ = make_blobs(n_samples=50_000, centers=8, n_features=8, random_state=SEED)
for estimator in [
    KMeans(n_clusters=8, n_init=3, random_state=SEED),
    MiniBatchKMeans(n_clusters=8, batch_size=1024, n_init=3, random_state=SEED),
]:
    start = time.perf_counter()
    estimator.fit(X_large)
    print(type(estimator).__name__,
          "seconds=", round(time.perf_counter() - start, 3),
          "inertia=", round(float(estimator.inertia_), 2))

## 11. GPU suitability and optional PyTorch implementation

The assignment step is dominated by large matrix operations and can parallelize well on a GPU. Yet transfer overhead, VRAM, and library implementation matter. This optional PyTorch version keeps the input on the selected device and chunks assignments. It uses random initialization for simplicity; it is not intended to replace mature GPU clustering libraries without benchmarking.

In [ ]:
try:
    import torch

    def kmeans_torch(X, n_clusters, max_iter=100, tol=1e-4, batch_size=8192, seed=42):
        """Educational PyTorch K-means; uses CUDA when available, otherwise CPU."""
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        X_t = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
        if X_t.ndim != 2 or X_t.shape[0] == 0:
            raise ValueError("X must be a non-empty 2D matrix")
        if not torch.isfinite(X_t).all():
            raise ValueError("X must contain finite values")
        n, d = X_t.shape
        if not 1 <= n_clusters <= n:
            raise ValueError("n_clusters must be in [1, n_samples]")
        generator = torch.Generator(device=device).manual_seed(seed)
        init_idx = torch.randperm(n, generator=generator, device=device)[:n_clusters]
        centers = X_t[init_idx].clone()
        x_sq = (X_t * X_t).sum(dim=1)

        for _ in range(max_iter):
            c_sq = (centers * centers).sum(dim=1)
            labels_parts, errors_parts = [], []
            for start in range(0, n, batch_size):
                xb = X_t[start:start + batch_size]
                dist = x_sq[start:start + batch_size, None] + c_sq[None, :] - 2 * xb @ centers.T
                dist = torch.clamp(dist, min=0)
                e, lab = dist.min(dim=1)
                labels_parts.append(lab)
                errors_parts.append(e)
            labels = torch.cat(labels_parts)
            errors = torch.cat(errors_parts)
            new_centers = centers.clone()
            for j in range(n_clusters):
                members = X_t[labels == j]
                if len(members):
                    new_centers[j] = members.mean(dim=0)
                else:
                    new_centers[j] = X_t[torch.argmax(errors)]
            shift = torch.linalg.vector_norm(new_centers - centers).item()
            centers = new_centers
            if shift <= tol:
                break

        # Report final assignments and objective.
        c_sq = (centers * centers).sum(dim=1)
        all_labels, all_errors = [], []
        for start in range(0, n, batch_size):
            xb = X_t[start:start + batch_size]
            dist = (X_t[start:start + batch_size] * X_t[start:start + batch_size]).sum(dim=1, keepdim=True) + c_sq[None, :] - 2 * xb @ centers.T
            dist = torch.clamp(dist, min=0)
            e, lab = dist.min(dim=1)
            all_labels.append(lab)
            all_errors.append(e)
        return centers.detach().cpu().numpy(), torch.cat(all_labels).cpu().numpy(), {
            "device": str(device), "inertia": float(torch.cat(all_errors).sum().item())
        }

    torch_centers, torch_labels, torch_info = kmeans_torch(X, 4, seed=SEED)
    print(torch_info)
except ImportError:
    print("PyTorch is not installed; skip the optional GPU section.")

## 12. Failure case: non-convex clusters

K-means partitions space into Voronoi regions around centroids. It cannot naturally represent two crescent shapes as separate clusters with a single centroid each. DBSCAN or spectral clustering may fit this geometry better, with different parameter choices and scaling costs.

In [ ]:
X_moons, y_moons = make_moons(n_samples=500, noise=0.06, random_state=SEED)
moon_centers, moon_labels, moon_info = kmeans_numpy(X_moons, 2, random_state=SEED)
print("K-means ARI on moons:", round(adjusted_rand_score(y_moons, moon_labels), 3))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=moon_labels, s=18)
plt.title("K-means on non-convex moons: a failure mode")
plt.show()

## 13. Correctness tests and synthesis

K-means alternates two minimization steps: nearest-centroid assignment and mean-centroid update. Each step does not increase within-cluster sum of squares, but the algorithm can settle at a local optimum. Use it when centroid-based compact clusters fit the task; choose another clustering family when geometry or density assumptions fail.

In [ ]:
C1, L1, I1 = kmeans_numpy(X, 4, random_state=SEED)
C2, L2, I2 = kmeans_numpy(X, 4, random_state=SEED)
assert np.allclose(C1, C2) and np.array_equal(L1, L2)
assert I1["inertia"] >= 0 and len(L1) == len(X)
assert np.isfinite(C1).all()

# Degenerate case: all points identical, repeated centers are acceptable.
X_same = np.ones((12, 2))
Cs, Ls, Is = kmeans_numpy(X_same, 3, random_state=SEED)
assert np.isfinite(Cs).all() and Is["inertia"] >= 0

try:
    kmeans_numpy(np.array([1, 2, 3]), 2)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for one-dimensional X")
print("All K-means checks passed.")